# D1-10 Optional - Database-level parameters and Monte Carlo

This optional notebook revisits the carbon-fiber foreground from Day 1, but changes where the uncertainty lives.

The Day 1 `assets/` folder contains a modified copy of the carbon-fiber LCI workbook: `lci-carbon-fiber-with-db-parameters.xlsx`.
The original Day 1 workbook has a small activity-level parameter block for `carbon fiber production, exhaust gas treatment 2`.
The modified workbook clears that activity-level block, adds a separate `Database parameters` sheet, and connects several exchange formulas to database-level parameters.

The final Monte Carlo samples normal Brightway exchange uncertainty and the new database-level parameter uncertainty for `carbon fiber production, weaved, at factory`.

## Learning goals

- Inspect a modified `bw2io` Excel foreground inventory with database-level parameters.
- Import the foreground database with active database parameters.
- Attach formula-bearing exchanges to a parameter recalculation group.
- Sample database-level parameter uncertainty with `bw2parameters.ParameterSet`.
- Use a small `bw_processing` datapackage to pass sampled formula amounts to Monte Carlo.

## 1) Setup

This notebook assumes Day 1 has already been run in the same Brightway project.
In particular, the `bafu` background database and the EF v3.1 climate method should already be available.

In [ ]:
from pathlib import Path

import bw2calc as bc
import bw2data as bd
import bw2io as bi
import bw_processing as bwp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from bw2data import parameters
from bw2data.parameters import ActivityParameter, DatabaseParameter, Group, ParameterizedExchange
from bw2parameters import ParameterSet

bd.projects.set_current('aalborg-rlcia-2026')

source_foreground = 'carbon fiber'
foreground_db = 'carbon fiber database parameters'
background_name = 'bafu'
biosphere_name = 'ecoinvent-3.10-biosphere'
formula_group = 'carbon_fiber_database_parameter_formulas'
activity_name = 'carbon fiber production, weaved, at factory'
method = ('EF v3.1', 'climate change', 'global warming potential (GWP100)')

parameter_workbook = Path('assets/lci-carbon-fiber-with-db-parameters.xlsx')

print('Current project:', bd.projects.current)
print('Database-parameter workbook:', parameter_workbook.resolve())
print('Target foreground database:', foreground_db)

## 2) Inspect the database-parameter workbook

The Day 1 source workbook is left untouched.
The prepared copy in this folder changes three things:

- the database name is `carbon fiber database parameters`
- the old activity-level `Parameters` block is cleared
- a new `Database parameters` worksheet is added, and selected exchanges have formulas that reference those parameters

The parameter choices are deliberately a little different from Day 1: they cover gas cleanup, shared high-temperature electricity demand, sizing resin, weaving scrap, weaving electricity, and weaving compressed air.

## 3) Check what the importer sees

Before writing anything to Brightway, inspect the parsed workbook.
The important checks are:

- database parameters are present
- no activity-level parameters remain
- the expected exchanges have formulas

In [ ]:
preview_importer = bi.ExcelImporter(str(parameter_workbook))

activity_parameter_blocks = [
    dataset['name']
    for dataset in preview_importer.data
    if dataset.get('parameters')
]

formula_preview = []
for dataset in preview_importer.data:
    for exchange in dataset.get('exchanges', []):
        if exchange.get('formula'):
            formula_preview.append(
                {
                    'activity': dataset['name'],
                    'exchange': exchange['name'],
                    'type': exchange.get('type'),
                    'amount': exchange.get('amount'),
                    'formula': exchange['formula'],
                }
            )

print('Workbook database name:', preview_importer.db_name)
print('Database parameters:', len(preview_importer.database_parameters or []))
print('Activity parameter blocks:', len(activity_parameter_blocks))
print('Formula exchanges:', len(formula_preview))

pd.DataFrame(formula_preview)

In [ ]:
pd.DataFrame(preview_importer.database_parameters)

## 4) Import the prepared foreground with activated database parameters

The import uses the same migration fixes as Day 1 and links the foreground to the existing `bafu` and biosphere databases.

After writing the database, we attach formula exchanges to one recalculation group. The uncertain variables themselves are database parameters; the group is only used so Brightway knows which exchanges have formulas to recalculate.

In [ ]:
migration_name = 'carbon-fiber-demo-fixes'

carbon_importer = bi.ExcelImporter(str(parameter_workbook))
carbon_importer.apply_strategies()
carbon_importer.data = bi.strategies.migrate_exchanges(
    db=carbon_importer.data,
    migration=migration_name,
)

carbon_importer.match_database(fields=('name', 'reference product', 'location', 'unit'))
carbon_importer.match_database(background_name, fields=('name', 'reference product', 'location', 'unit'))
carbon_importer.match_database(
    biosphere_name,
    fields=('name', 'unit', 'categories'),
    relink=True,
    edge_kinds=['biosphere'],
)

print('Import statistics before writing:')
carbon_importer.statistics()

if list(carbon_importer.unlinked):
    raise RuntimeError('The prepared carbon-fiber workbook still has unlinked exchanges. Inspect carbon_importer.unlinked.')

# Make the notebook rerunnable: remove old formula links for this optional D1-10 group.
ParameterizedExchange.delete().where(ParameterizedExchange.group == formula_group).execute()
ActivityParameter.delete().where(ActivityParameter.group == formula_group).execute()
Group.delete().where(Group.name == formula_group).execute()

carbon_importer.write_database(db_name=foreground_db, activate_parameters=True)

In [ ]:
formula_activity_names = [
    'carbon fiber production, exhaust gas treatment 2',
    'carbon fiber production, fiber stabilization, carbonization, electrolysis and washing',
    'carbon fiber production, fiber drying and sizing',
    'carbon fiber production, weaved, at factory',
]

formula_activities = []
for name in formula_activity_names:
    matches = [
        activity for activity in bd.Database(foreground_db)
        if activity['name'] == name and activity.get('location') == 'RER'
    ]
    if len(matches) != 1:
        raise ValueError(f'Expected one activity named {name!r}, found {len(matches)}')
    formula_activities.append(matches[0])

for activity in formula_activities:
    parameters.add_exchanges_to_group(formula_group, activity.key)

ActivityParameter.recalculate(formula_group)
ActivityParameter.recalculate_exchanges(formula_group)

print('Registered formula exchanges:', ParameterizedExchange.select().where(ParameterizedExchange.group == formula_group).count())
print('Database parameters:', DatabaseParameter.select().where(DatabaseParameter.database == foreground_db).count())

## 5) Inspect the active database parameters and formulas

The central formula amounts should be close to the original workbook values.
For example, the weaving scrap formula changes the input of dried and sized fiber from exactly `1` to `1 / (1 - 0.04)`.

In [ ]:
database_parameter_table = pd.DataFrame(
    [
        {
            'name': parameter.name,
            'amount': parameter.amount,
            'uncertainty type': parameter.data.get('uncertainty type'),
            'loc': parameter.data.get('loc'),
            'scale': parameter.data.get('scale'),
            'minimum': parameter.data.get('minimum'),
            'maximum': parameter.data.get('maximum'),
            'comment': parameter.data.get('comment'),
        }
        for parameter in DatabaseParameter.select().where(DatabaseParameter.database == foreground_db)
    ]
).sort_values('name').reset_index(drop=True)

database_parameter_table

In [ ]:
formula_rows = []
for activity in formula_activities:
    for exchange in activity.exchanges():
        if 'formula' in exchange:
            formula_rows.append(
                {
                    'sample column': f'exchange_{exchange._document.id}',
                    'activity': activity['name'],
                    'exchange': exchange.input['name'],
                    'type': exchange['type'],
                    'unit': exchange.get('unit'),
                    'central amount': float(exchange['amount']),
                    'formula': exchange['formula'],
                    'input id': exchange.input.id,
                    'output id': activity.id,
                    'matrix': 'biosphere_matrix' if exchange['type'] == 'biosphere' else 'technosphere_matrix',
                }
            )

formula_exchange_table = pd.DataFrame(formula_rows)
formula_exchange_table[
    ['activity', 'exchange', 'type', 'central amount', 'unit', 'formula']
]

## 6) Deterministic score

This is the score after evaluating the database-level formulas at their central parameter values.

In [ ]:
target_matches = [
    activity for activity in bd.Database(foreground_db)
    if activity['name'] == activity_name
    and activity.get('location') == 'RER'
    and activity.get('unit') == 'kilogram'
]
if len(target_matches) != 1:
    raise ValueError(f'Expected one target activity in {foreground_db}, found {len(target_matches)}')
target_activity = target_matches[0]


def calculate_score(activity):
    lca = bc.LCA({activity: 1}, method=method)
    lca.lci()
    lca.lcia()
    return float(lca.score)


score_rows = [
    {
        'database': foreground_db,
        'activity': target_activity['name'],
        'score': calculate_score(target_activity),
    }
]

if source_foreground in bd.databases:
    original_matches = [
        activity for activity in bd.Database(source_foreground)
        if activity['name'] == activity_name
        and activity.get('location') == 'RER'
        and activity.get('unit') == 'kilogram'
    ]
    if len(original_matches) == 1:
        score_rows.insert(
            0,
            {
                'database': source_foreground,
                'activity': original_matches[0]['name'],
                'score': calculate_score(original_matches[0]),
            },
        )

pd.DataFrame(score_rows)

## 7) Sample database parameters

Brightway stores the database parameters, but `bw2calc` Monte Carlo does not automatically sample parameter uncertainty and push it into formula exchanges.
We therefore sample the database parameters with `ParameterSet` and evaluate the exchange formulas for each draw.

The sampled formula amounts will later be supplied to Brightway as absolute replacement arrays for the formula exchanges.
The rest of the database can still use normal Brightway exchange uncertainty.

In [ ]:
iterations = 500
seed = 2026
np.random.seed(seed)

parameter_definitions = {}
for parameter in DatabaseParameter.select().where(DatabaseParameter.database == foreground_db):
    parameter_definitions[parameter.name] = {'amount': parameter.amount, **parameter.data}

exchange_formula_definitions = {}
for _, row in formula_exchange_table.iterrows():
    exchange_formula_definitions[row['sample column']] = {
        'amount': row['central amount'],
        'formula': row['formula'],
    }

sampler = ParameterSet({**parameter_definitions, **exchange_formula_definitions})
parameter_samples = sampler.evaluate_monte_carlo(iterations=iterations)

sampled_parameter_table = pd.DataFrame(
    {name: parameter_samples[name] for name in parameter_definitions}
).describe(percentiles=[0.05, 0.5, 0.95]).T[
    ['mean', 'std', '5%', '50%', '95%']
]

sampled_parameter_table

In [ ]:
sampled_exchange_table = pd.DataFrame(
    {
        row['sample column']: parameter_samples[row['sample column']]
        for _, row in formula_exchange_table.iterrows()
    }
).describe(percentiles=[0.05, 0.5, 0.95]).T[
    ['mean', 'std', '5%', '50%', '95%']
]

sampled_exchange_table = sampled_exchange_table.join(
    formula_exchange_table.set_index('sample column')[['activity', 'exchange', 'central amount', 'formula']]
)
sampled_exchange_table[['activity', 'exchange', 'central amount', 'mean', 'std', '5%', '50%', '95%', 'formula']]

## 8) Build a parameter datapackage

The datapackage supplies sampled absolute amounts for the exchanges controlled by database-level parameters.
These values replace the central amounts for those same matrix coordinates during Monte Carlo.

For the parameterized exchanges only, we use parameter samples instead of their original exchange uncertainty. Other exchanges in the database still use normal Brightway uncertainty.

In [ ]:
parameter_override_dp = bwp.create_datapackage(sequential=True)

for matrix_name in ['technosphere_matrix', 'biosphere_matrix']:
    subset = formula_exchange_table[formula_exchange_table['matrix'] == matrix_name]
    if subset.empty:
        continue

    indices_array = np.array(
        list(zip(subset['input id'].astype(int), subset['output id'].astype(int))),
        dtype=bwp.INDICES_DTYPE,
    )

    data_array = np.vstack(
        [parameter_samples[row['sample column']] for _, row in subset.iterrows()]
    )

    kwargs = {
        'matrix': matrix_name,
        'indices_array': indices_array,
        'data_array': data_array,
        'name': f'{matrix_name}-database-parameter-overrides',
    }

    # Technosphere inputs need the normal Brightway sign convention.
    if matrix_name == 'technosphere_matrix':
        kwargs['flip_array'] = np.ones(len(subset), dtype=bool)

    parameter_override_dp.add_persistent_array(**kwargs)

parameter_override_dp

## 9) Monte Carlo for woven carbon fiber

The first run uses ordinary Brightway exchange uncertainty.
The second run uses the same database, but replaces the database-parameterized formula exchanges with sampled absolute amounts.

In [ ]:
method in bd.methods

In [ ]:
def monte_carlo_scores(activity, use_parameter_overrides=False):
    demand, data_objs, remapping = bd.prepare_lca_inputs(
        demand={activity: 1},
        method=method,
    )

    if use_parameter_overrides:
        data_objs = data_objs + [parameter_override_dp]

    lca = bc.LCA(
        demand,
        data_objs=data_objs,
        remapping_dicts=remapping,
        use_distributions=True,
        use_arrays=use_parameter_overrides,
        seed_override=seed,
    )
    lca.lci()
    lca.lcia()

    scores = [float(lca.score)]
    for _ in range(iterations - 1):
        next(lca)
        scores.append(float(lca.score))
    return np.array(scores)


exchange_only_scores = monte_carlo_scores(target_activity, use_parameter_overrides=False)
parameterized_scores = monte_carlo_scores(target_activity, use_parameter_overrides=True)

scores = pd.DataFrame(
    {
        'exchange uncertainty only': exchange_only_scores,
        'exchange uncertainty with database parameter overrides': parameterized_scores,
    }
)

summary = scores.describe(percentiles=[0.05, 0.5, 0.95]).T[
    ['mean', 'std', '5%', '50%', '95%']
]
summary['p95 - p05'] = summary['95%'] - summary['5%']
summary

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.hist(
    scores['exchange uncertainty only'],
    bins=40,
    alpha=0.6,
    label='exchange uncertainty only',
)
plt.hist(
    scores['exchange uncertainty with database parameter overrides'],
    bins=40,
    alpha=0.6,
    label='exchange uncertainty with database parameter overrides',
)
plt.xlabel(f"Climate score [{bd.Method(method).metadata.get('unit', 'impact units')}]")
plt.ylabel('Frequency')
plt.title('Carbon fiber production, weaved, at factory')
plt.legend()
plt.tight_layout()
plt.show()

## Recap

In this notebook you used a copied foreground workbook with database-level parameters, imported it with active parameters, attached exchange formulas to a recalculation group, and ran a Monte Carlo that includes sampled database-level parameter uncertainty.

The important modeling distinction is that database parameters are useful for assumptions shared across several activities, while activity parameters are better for local assumptions inside one activity.